# Dogs vs. Cats Image Classification — EfficientNetB4 Transfer Learning

**Competition:** Dogs vs. Cats Redux: Kernels Edition  
**Platform:** Google Colab (GPU T4)  
**Model:** EfficientNetB4 @ 260px — Two-Phase Transfer Learning

---

⚠️ **BEFORE RUNNING:**
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run Step 0 first and upload your `kaggle.json` when prompted
3. Then **Run All**

---

## Step 0 — Kaggle Setup & Data Download

Download competition data using Kaggle API. Upload your `kaggle.json` when prompted.

In [ ]:
import os, shutil, zipfile

os.system('pip install -q kaggle')

from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file...')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json configured.')

if not os.path.exists('/content/dogs-vs-cats-redux-kernels-edition.zip'):
    print('Downloading competition data...')
    os.system('kaggle competitions download -c dogs-vs-cats-redux-kernels-edition -p /content')
    print('Download complete!')

if not os.path.exists('/content/train_raw'):
    print('Extracting...')
    with zipfile.ZipFile('/content/dogs-vs-cats-redux-kernels-edition.zip','r') as z:
        z.extractall('/content/')
    with zipfile.ZipFile('/content/train.zip','r') as z:
        z.extractall('/content/train_raw')
    with zipfile.ZipFile('/content/test.zip','r') as z:
        z.extractall('/content/test_raw')
    print('Extraction complete!')

print('Step 0 done!')

---

## Step 1 — Imports & Environment

In [ ]:
import warnings
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import tensorflow        as tf

from sklearn.metrics                             import log_loss
from tensorflow.keras                            import layers, models, callbacks
from tensorflow.keras.applications               import EfficientNetB4
from tensorflow.keras.applications.efficientnet  import preprocess_input
from tensorflow.keras.preprocessing.image        import ImageDataGenerator

warnings.filterwarnings('ignore')

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

---

## Step 2 — Organise Images into Class Subfolders

`flow_from_directory` requires one subfolder per class.

In [ ]:
CAT_DIR  = '/content/data/train/cats'
DOG_DIR  = '/content/data/train/dogs'
TEST_OUT = '/content/test'

os.makedirs(CAT_DIR,  exist_ok=True)
os.makedirs(DOG_DIR,  exist_ok=True)
os.makedirs(TEST_OUT, exist_ok=True)

if len(os.listdir(CAT_DIR)) == 0:
    print('Organising training images...')
    src = '/content/train_raw/train'
    for fname in os.listdir(src):
        if   fname.startswith('cat'): shutil.copy(os.path.join(src,fname), CAT_DIR)
        elif fname.startswith('dog'): shutil.copy(os.path.join(src,fname), DOG_DIR)
    print(f'Cats: {len(os.listdir(CAT_DIR))} | Dogs: {len(os.listdir(DOG_DIR))}')
else:
    print('Training images already organised.')

if len(os.listdir(TEST_OUT)) == 0:
    print('Organising test images...')
    src = '/content/test_raw/test'
    for fname in os.listdir(src):
        shutil.copy(os.path.join(src,fname), TEST_OUT)
    print(f'Test images: {len(os.listdir(TEST_OUT))}')
else:
    print('Test images already organised.')

---

## Step 3 — Data Generators & Augmentation

**Preprocessing:** EfficientNetB4 `preprocess_input` — channel-wise mean subtraction, NOT rescale/255.  
**Augmentation:** Training only. Validation and test never augmented.

| Technique | Value |
|-----------|-------|
| Horizontal flip | True |
| Rotation | ±20° |
| Zoom | ±20% |
| Brightness | 0.8–1.2 |

In [ ]:
IMG_SIZE   = 260
BATCH_SIZE = 32
VAL_SPLIT  = 0.20
SEED       = 42
TRAIN_DIR  = '/content/data/train'

train_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = VAL_SPLIT,
    horizontal_flip        = True,
    rotation_range         = 20,
    zoom_range             = 0.20,
    brightness_range       = [0.8, 1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split       = VAL_SPLIT
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary',
    subset='training', seed=SEED)

val_gen = val_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary',
    subset='validation', shuffle=False, seed=SEED)

print(f'Training images   : {train_gen.samples}')
print(f'Validation images : {val_gen.samples}')
print(f'Class mapping     : {train_gen.class_indices}')
print('Expected          : cats=0, dogs=1')

---

## Step 4 — Build EfficientNetB4 Model

EfficientNetB4 pretrained on ImageNet. Custom binary classification head.  
Backbone frozen for Phase 1.

In [ ]:
base_model = EfficientNetB4(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

x   = base_model.output
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.5)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(base_model.input, out)

total     = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Total backbone layers : {len(base_model.layers)}')
print(f'Total parameters      : {total:,}')
print(f'Trainable (head only) : {trainable:,}')

---

## Step 5 — Phase 1: Train Head Only

Backbone frozen. Only classification head trained at LR=1e-3.

In [ ]:
print('=== PHASE 1 ===')

model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-3),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

cb1 = [
    callbacks.ModelCheckpoint('/content/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1)
]

history1 = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=cb1)

p1_best_acc  = max(history1.history['val_accuracy'])
p1_best_loss = min(history1.history['val_loss'])
print(f'\nPhase 1 complete.')
print(f'Best val accuracy : {p1_best_acc*100:.2f}%')
print(f'Best val loss     : {p1_best_loss:.4f}')

---

## Step 6 — Phase 2: Fine-Tune Top 100 Layers

Unfreeze top 100 backbone layers. Fine-tune at LR=1e-5.

In [ ]:
print('=== PHASE 2 ===')

for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen backbone layers: {unfrozen} of {len(base_model.layers)}')

model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-5),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

cb2 = [
    callbacks.ModelCheckpoint('/content/best_phase2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.3, verbose=1)
]

history2 = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=cb2)

p2_best_acc  = max(history2.history['val_accuracy'])
p2_best_loss = min(history2.history['val_loss'])
print(f'\nPhase 2 complete.')
print(f'Best val accuracy : {p2_best_acc*100:.2f}%')
print(f'Best val loss     : {p2_best_loss:.4f}')
print(f'Improvement       : +{(p2_best_acc - p1_best_acc)*100:.2f}%')

---

## Step 7 — Training History Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EfficientNetB4 @ 260px — Phase 1 & Phase 2 Training History')

e1 = len(history1.history['loss'])
e2 = len(history2.history['loss'])
x1 = range(1, e1+1)
x2 = range(e1+1, e1+e2+1)

ax1.plot(x1, history1.history['loss'],     'b-',  label='Train Loss')
ax1.plot(x1, history1.history['val_loss'], 'r--', label='Val Loss')
ax1.plot(x2, history2.history['loss'],     'b-')
ax1.plot(x2, history2.history['val_loss'], 'r--')
ax1.axvline(x=e1+0.5, color='gray', linestyle=':', label='Phase 1→2')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(x1, history1.history['accuracy'],     'b-',  label='Train Acc')
ax2.plot(x1, history1.history['val_accuracy'], 'r--', label='Val Acc')
ax2.plot(x2, history2.history['accuracy'],     'b-')
ax2.plot(x2, history2.history['val_accuracy'], 'r--')
ax2.axvline(x=e1+0.5, color='gray', linestyle=':', label='Phase 1→2')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Step 8 — Generate Submission CSV

Predict on 12,500 test images. Auto-downloads submission.csv to your computer.

In [ ]:
model.load_weights('/content/best_phase2.keras')
print('Best Phase 2 weights loaded.')

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = test_datagen.flow_from_directory(
    '/content/', classes=['test'],
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode=None, shuffle=False)

preds = model.predict(test_gen, verbose=1)
ids   = [int(os.path.basename(f).split('.')[0]) for f in test_gen.filenames]
sub   = pd.DataFrame({'id': ids, 'label': preds.flatten()}).sort_values('id')
sub.to_csv('/content/submission.csv', index=False)

print(f'\n✅ submission.csv saved!')
print(f'Total rows : {len(sub)}')
print(f'Min pred   : {sub.label.min():.6f}')
print(f'Max pred   : {sub.label.max():.6f}')
print(f'Mean pred  : {sub.label.mean():.4f}')
print(sub.head(10))

from google.colab import files
files.download('/content/submission.csv')

---

## Step 9 — Validation Score Estimate

In [ ]:
val_gen.reset()
val_preds  = model.predict(val_gen, verbose=1)
val_labels = val_gen.classes
val_ll     = log_loss(val_labels, val_preds)
val_acc    = ((val_preds > 0.5).flatten() == val_labels).mean()

print('Validation Performance Estimate')
print('='*35)
print(f'Log Loss : {val_ll:.5f}')
print(f'Accuracy : {val_acc*100:.2f}%')

score_table = [(0.050,40),(0.055,39),(0.060,38),(0.065,37),
               (0.070,36),(0.075,35),(0.080,34),(0.100,33),(0.200,31),(0.500,30)]
print('\nExpected Kaggle points:')
for threshold, points in score_table:
    marker = '  <-- estimated position' if val_ll < threshold else ''
    print(f'  log loss < {threshold:.3f} : {points} pts{marker}')

---

## Step 10 — Submission Instructions

1. `submission.csv` auto-downloaded to your **Downloads folder**
2. Go to Kaggle competition → **Submit Predictions**
3. Upload `submission.csv`
4. Wait ~2 min for score
5. Check the Private Score under My Submissions